### 1. Prepare pdbqt for 8vb5 and a1aac(as native ligand)

In [1]:
import os
import re
import numpy as np
import pandas as pd
import urllib.request
from glob import glob
from pathlib import Path

# from Bio.PDB import MMCIFParser, PDBParser, PDBIO, Select

In [15]:
BASE_DIR = Path("/home/ssm-user/project/autodock")
BASE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE_DIR)
print("BASE_DIR is set to:", BASE_DIR)

BASE_DIR is set to: /home/ssm-user/project/autodock


In [15]:
# pdb_id = "8vb5"
# cif = f"{pdb_id.upper()}.cif"
# pdb = f"{pdb_id.lower()}.pdb"

# urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id.upper()}.cif", str(cif))
# parser = MMCIFParser()
# structure = parser.get_structure(pdb_id.upper(), str(cif))
# io = PDBIO()
# io.set_structure(structure)
# io.save(str(pdb))

# print("Saved PDB:", pdb)

Saved PDB: 8vb5.pdb


In [ ]:
# # in bash
# wget https://sw-tools.rcsb.org/apps/MAXIT/maxit-v11.300-prod-src.tar.gz
# tar -xvzf maxit-v11.300-prod-src.tar.gz
# maxit -input /home/ssm-user/project/autodock/8VB5.cif -output /home/ssm-user/project/autodock/8vb5.pdb -o PDB

In [3]:
residues = set()

with open('8vb5.pdb', 'r') as f:
    for l in f:
        if l.startswith('HETATM'):
            residues.add((l[17:22], l[21:24]))

for resname, chain in residues:
    print(resname, chain)

HOH A A13
HOH A A14
HOH A A12
EDO A A11
AAC A A11
PEG A A11
 CL A A11


In [4]:
protein_file = "8vb5.pdb"
output_protein_file = "8vb5_prot.pdb"
ligand_file = "a1aac.pdb"
# fixed_ligand_file = "a1aac_fixed.pdb"

In [5]:
# ------------------------
# Step 1: Prepare protein for pdb2pqr
# ------------------------
with open(output_protein_file, 'w') as g:
    with open(protein_file, 'r') as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                resname = line[17:20].strip()
                if line.startswith('ATOM') and line[21] == 'A':

                    res_num_str = line[22:26].strip()
                    res_num_pure = ''.join(filter(str.isdigit, res_num_str))

                    new_line = list(line)
                    new_line[22:26] = f"{res_num_pure:>4}"
                    g.write("".join(new_line))
                elif line.startswith('HETATM') and resname not in ['EDO', 'PEG']:
                    g.write(line)
            elif line.startswith('TER'):
                g.write('TER\n')
        g.write('END\n')
print(f"Protein file created: {output_protein_file}")

Protein file created: 8vb5_prot.pdb


In [6]:
# ------------------------
# Step 2: Extract ligand (A1AAC)
# ------------------------
with open(os.path.join(BASE_DIR, "a1aac.pdb"), "w") as g:
    with open(protein_file, 'r') as f:
        for line in f:
            if line.startswith('HETATM') and line[17:20].strip() == "AAC":
                g.write(line)
        g.write("END\n")

# ------------------------
# Step 3: Fix ligand PDB for AutoDock
# ------------------------
# with open(fixed_ligand_file, "w") as out_file:
#     with open(ligand_file, "r") as in_file:
#         for line in in_file:
#             if line.startswith("HETATM"):
#                 fixed_line = list(line)
#                 fixed_line[17:20] = ['A','A','C']
#                 fixed_line[21] = 'A'
#                 fixed_line[22:26] = list(f"{1:>4}")
#                 atom_name = line[12:16].strip()
#                 if 'C' in atom_name:
#                     fixed_line[76:78] = [' ', 'C']
#                 elif 'N' in atom_name:
#                     fixed_line[76:78] = [' ', 'N']
#                 elif 'O' in atom_name:
#                     fixed_line[76:78] = [' ', 'O']
#                 fixed_line[78:80] = [' ', ' ']
#                 out_file.write("".join(fixed_line))
#             else:
#                 out_file.write(line)
# print(f"Fixed ligand file created: {fixed_ligand_file}")


In [7]:
# ------------------------
# Step 4: Calculate geometric center of ligand
# ------------------------
ligand_geom = []
coord_pattern = re.compile(r'HETATM\s+.*?\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)')
with open(ligand_file, 'r') as f:
    for line in f:
        m = coord_pattern.match(line)
        if m:
            x, y, z = [float(c) for c in m.groups()]
            ligand_geom.append([x, y, z])

if ligand_geom:
    ligand_geom = np.array(ligand_geom)
    center = ligand_geom.mean(axis=0)
    print(f"Geometric Center of ligand: {center[0]:.3f} {center[1]:.3f} {center[2]:.3f}")
else:
    print("Warning: No coordinates could be extracted. Check the input file format.")

Geometric Center of ligand: -1.128 6.733 -18.115


In [ ]:
# pdb2pqr: AMBER99ff parameter (pH 7.4)
! /home/ssm-user/miniforge3/envs/docking/bin/pdb2pqr30 \
    --ff AMBER \
    --keep-chain \
    --titration-state-method propka \
    --with-ph 7.4 \
    8vb5_prot.pdb \
    8vb5_prot.pqr

In [9]:
# pqr -> pdbqt
! /home/ssm-user/apps/mgltools/bin/pythonsh \
    /home/ssm-user/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_receptor4.py \
    -r 8vb5_prot.pqr \
    -o 8vb5_prot.pdbqt \
    -C \
    -U nphs_lps \
    -v

setting PYTHONHOME environment
set verbose to  True
read  8vb5_prot.pqr
setting up RPO with mode= automatic and outputfilename=  8vb5_prot.pdbqt
charges_to_add= None
delete_single_nonstd_residues= None


In [10]:
# native ligand: pdb -> mol2
! obabel -ipdb a1aac.pdb -omol2 -O a1aac.mol2

1 molecule converted


In [11]:
# mol2 -> pdbqt
! /home/ssm-user/apps/mgltools/bin/pythonsh \
    /home/ssm-user/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py \
    -l a1aac.mol2 \
    -o a1aac.pdbqt \
    -U nphs_lps \
    -v

setting PYTHONHOME environment
set verbose to  True
read  a1aac.mol2
setting up LPO with mode= automatic and outputfilename=  a1aac.pdbqt
and check_for_fragments= False
and bonds_to_inactivate= 
returning  0
No change in atomic coordinates


### 2. Native ligand(a1aac) based AutoGrid
---
all possible atom types should be specified for virtual screening since we will handle diverse molecules.

AutoDock Atom types:
https://autodock.scripps.edu/wp-content/uploads/sites/31/2019/03/AD4.1_bound.dat

In [17]:
%%bash
cd /home/ssm-user/project/autodock
~/apps/mgltools/bin/pythonsh ~/apps/mgltools/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_gpf4.py \
    -l a1aac.pdbqt \
    -r 8vb5_prot.pdbqt \
    -y \
    -p ligand_types='H,HD,A,C,N,NA,NS,OA,OS,F,Cl,Br,S,SA' \
    -p npts='80,80,80'

setting PYTHONHOME environment
setting ligand_types: newvalue= H HD A C N NA NS OA OS F Cl Br S SA


In [18]:
%%bash
cd /home/ssm-user/project/autodock
/home/ssm-user/miniforge3/envs/docking/bin/autogrid4 -p 8vb5_prot.gpf -l 8vb5_prot.glg
ls -la *.fld *.map
cd ../

-rw-r--r-- 1 ssm-user ssm-user 4114935 Aug 18 15:15 8vb5_prot.A.map
-rw-r--r-- 1 ssm-user ssm-user 4286075 Aug 18 15:15 8vb5_prot.Br.map
-rw-r--r-- 1 ssm-user ssm-user 4123625 Aug 18 15:15 8vb5_prot.C.map
-rw-r--r-- 1 ssm-user ssm-user 4201643 Aug 18 15:15 8vb5_prot.Cl.map
-rw-r--r-- 1 ssm-user ssm-user 3891147 Aug 18 15:15 8vb5_prot.F.map
-rw-r--r-- 1 ssm-user ssm-user 3646053 Aug 18 15:15 8vb5_prot.H.map
-rw-r--r-- 1 ssm-user ssm-user 3696438 Aug 18 15:15 8vb5_prot.HD.map
-rw-r--r-- 1 ssm-user ssm-user 4034773 Aug 18 15:15 8vb5_prot.N.map
-rw-r--r-- 1 ssm-user ssm-user 4041111 Aug 18 15:15 8vb5_prot.NA.map
-rw-r--r-- 1 ssm-user ssm-user 4041289 Aug 18 15:15 8vb5_prot.NS.map
-rw-r--r-- 1 ssm-user ssm-user 4019460 Aug 18 15:15 8vb5_prot.OA.map
-rw-r--r-- 1 ssm-user ssm-user 4019460 Aug 18 15:15 8vb5_prot.OS.map
-rw-r--r-- 1 ssm-user ssm-user 4158929 Aug 18 15:15 8vb5_prot.S.map
-rw-r--r-- 1 ssm-user ssm-user 4175413 Aug 18 15:15 8vb5_prot.SA.map
-rw-r--r-- 1 ssm-user ssm-user 3182085 A

### 3. Get pdbqt from potent smiles(Enamine)

In [20]:
!mkdir -p /home/ssm-user/project/autodock/potent
!mkdir -p /home/ssm-user/project/autodock/potent_pdbqt

### [Bash] Run "03_make_smi_from_csv.py"
---
python 03_make_smi_from_csv.py /home/ssm-user/project/smiles_batch_2_20029.csv /home/ssm-user/project/autodock/potent

In [21]:
potent_path = Path("/home/ssm-user/project/autodock/potent")
smi_files = list(potent_path.glob("*.smi"))
print(f"Created .smi: {len(smi_files)}")

Created .smi: 20029


In [23]:
cpu_count = os.cpu_count()
print(f"avaliable CPU cores: {cpu_count}")

avaliable CPU cores: 8


### [Bash] Run "04_prepare_pdbqt_parallel.sh"
---
chmod +x 04_prepare_pdbqt_parallel.sh

export JOBS=8

./04_prepare_pdbqt_parallel.sh

In [32]:
potent_pdbqt_path = Path("/home/ssm-user/project/autodock/potent_pdbqt")
pdbqt_files = list(potent_pdbqt_path.glob("*.pdbqt"))
print(f"Created .pdbqt: {len(pdbqt_files)}")

Created .pdbqt: 1760


### 4. Making a batch file
---------
https://github.com/ccsb-scripps/AutoDock-GPU

In [31]:
list_path = Path("/home/ssm-user/project/autodock/ligand_list.txt")
with open(list_path, "w") as f:
    f.write("./8vb5_prot.maps.fld\n")
print(f"{list_path}")

/home/ssm-user/project/autodock/ligand_list.txt


### 5. Run AutoDock

In [33]:
with open('/home/ssm-user/project/autodock/ligand_list.txt', 'w') as fout:
    fout.write('./8vb5_prot.maps.fld \n') 
    files = glob('./potent_pdbqt/*.pdbqt')
    
    for filename in files:
        fout.write(os.path.abspath(filename) + '\n')
        fout.write(filename.split('/')[-1].split('.')[0] + '\n')

In [ ]:
%%bash
cd /home/ssm-user/project/autodock
rm -f *.dlg *.xml
 
~/apps/AutoDock-GPU/bin/autodock_gpu_128wi -B ./ligand_list.txt | tee gpu_out

AutoDock-GPU version: v1.6-7-ga46ab564d2ac5f1a1523f65239b43505a1c29364-dirty

Using 8 OpenMP threads

Running 1975 docking calculations

Cuda device:                              Tesla T4
Available memory on device:               14810 MB (total: 14913 MB)

CUDA Setup time 0.702732s
(Thread 7 is setting up Job #1)
(Thread 2 is setting up Job #2)
(Thread 0 is setting up Job #3)
(Thread 1 is setting up Job #4)
(Thread 4 is setting up Job #5)
(Thread 3 is setting up Job #6)
(Thread 6 is setting up Job #7)
(Thread 5 is setting up Job #8)

Running Job #1:
    Device: Tesla T4
    Grid map file: ./8vb5_prot.maps.fld
    Ligand file: /home/ssm-user/project/autodock/potent_pdbqt/Z1260135885.pdbqt
    Output file: Z1260135885.dlg (+ xml)
    Using heuristics: (capped) number of evaluations set to 1132076
    Local-search chosen method is: ADADELTA (ad)

Rest of Setup time 0.095609s

Executing docking runs, stopping automatically after either reaching 0.15 kcal/mol standard deviation of
the best

### Analyze gpu_out
----------------------------

In [46]:
def get_min_affinity(dlg_file):
    with open(dlg_file, 'r') as f:
        lines = f.readlines()

    affinity_list = []
    for line in lines:
        if 'Estimated Free Energy of Binding' in line:
            try:
                affinity = float(line.strip().split()[-3])
                affinity_list.append(affinity)
            except (ValueError, IndexError):
                continue
    
    if affinity_list:
        return min(affinity_list)
    else:
        return None

In [47]:
dlg_files = glob.glob('/home/ssm-user/project/autodock/*.dlg') 
lig_and_energy = []

print("Processing dlg files...")
for f in dlg_files:
    min_e = get_min_affinity(f)
    if min_e is not None: 
        lig_name = os.path.basename(f).split('.')[0]  
        lig_and_energy.append((lig_name, min_e))

lig_and_energy.sort(key=lambda x: x[1])

print(f"\n=== TOP LIGANDS (Total: {len(lig_and_energy)}) ===")
print(f"{'Rank':<5} {'Ligand':<25} {'Binding Affinity (kcal/mol)':<25}")
print("-" * 60)

for rank, (ligand, energy) in enumerate(lig_and_energy, 1):
    print(f"{rank:<5} {ligand:<25} {energy:<25.2f}")

print(f"\n=== TOP 20 ===")
for rank, (ligand, energy) in enumerate(lig_and_energy[:20], 1):
    print(f"{rank}. {ligand}: {energy:.2f} kcal/mol")

Processing dlg files...

=== TOP LIGANDS (Total: 0) ===
Rank  Ligand                    Binding Affinity (kcal/mol)
------------------------------------------------------------

=== TOP 10 ===


In [ ]:
name2smi = {}
try:
    with open('./drugs.txt', 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                name2smi[parts[0]] = parts[2]
except FileNotFoundError:
    print("SMILES file not found, proceeding without SMILES")
    name2smi = {}

# 그 다음 원래 코드 실행
with open('/home/ssm-user/project/autodock/ligand_and_affinity.txt', 'w') as fout:
    fout.write('Rank\tLigand\tSMILES\tBinding_Affinity(kcal/mol)\n')
    
    for rank, (lig, ene) in enumerate(lig_and_energy, 1):
        smiles = name2smi.get(lig, 'N/A')  # SMILES가 없으면 'N/A'
        fout.write(f'{rank}\t{lig}\t{smiles}\t{ene:10.2f}\n')

In [ ]:
!cat /home/ssm-user/project/autodock/ligand_and_affinity.txt

In [ ]:
!obabel -ad -ipdbqt ./virtual_screening/Acetohexamide.dlg  -opdbqt -O ./virtual_screening/Acetohexamide.docked.pdbqt